# Lab 4 – Logical Reasoning for Planning
**Logic + Search = Planning.** Run cells top to bottom. Sections follow Tasks 0–8 of the lab sheet.

## Task 0 – The planning problem
- **(a) Initial state I** = `{At(Robot,A), At(Package,A)}`
- **(b) Goal G** = `{At(Package,C)}`
- **(c) Actions:** `Move(x,y)` for connected pairs (A-B, B-A, B-C, C-B), `PickUp(Package,l)`, `Drop(Package,l)`.
- **(d) Preconditions / effects**

| Action | Preconditions | Add effects | Delete effects |
|---|---|---|---|
| `Move(x,y)` | At(Robot,x) | At(Robot,y) | At(Robot,x) |
| `PickUp(Package,l)` | At(Robot,l), At(Package,l) | Holding(Package) | At(Package,l) |
| `Drop(Package,l)` | At(Robot,l), Holding(Package) | At(Package,l) | Holding(Package) |

**Applicability in I:**
- `PickUp(Package,A)`: needs At(Robot,A) ✔ and At(Package,A) ✔ → **applicable**.
- `Drop(Package,C)`: needs At(Robot,C) ✘ and Holding(Package) ✘ → **not applicable**.

## Task 1 – Plan by hand
| State | Facts | After action |
|---|---|---|
| S0 | At(Robot,A), At(Package,A) | – |
| S1 | At(Robot,A), Holding(Package) | PickUp(Package,A) |
| S2 | At(Robot,B), Holding(Package) | Move(A,B) |
| S3 | At(Robot,C), Holding(Package) | Move(B,C) |
| S4 | At(Robot,C), At(Package,C) | Drop(Package,C) |

**Note:** the lab sheet's hint (`Move(A,B)`, `PickUp(Package,B)`, ...) is *not* valid. The package is at A,
so `PickUp(Package,B)` needs `At(Package,B)`, which is false. The robot must pick the package up **at A first**.
The code below proves this.

## Task 2 – Planner (states = sets of propositions, BFS)

In [ ]:
from collections import deque, namedtuple

# An action = name + positive/negative preconditions + positive/negative effects
Action = namedtuple("Action", "name pos_pre neg_pre pos_eff neg_eff")

def make_action(name, pos_pre=(), neg_pre=(), pos_eff=(), neg_eff=()):
    return Action(name, frozenset(pos_pre), frozenset(neg_pre),
                  frozenset(pos_eff), frozenset(neg_eff))

def applicable(state, a):
    """LOGIC: state |= Preconditions(a)."""
    return a.pos_pre <= state and not (a.neg_pre & state)

def apply_action(state, a):
    """Remove negative effects first, then add positive effects."""
    return frozenset((state - a.neg_eff) | a.pos_eff)

def bfs_plan(initial, goal, actions):
    """SEARCH: breadth-first over states. Returns (plan, states) or (None, None)."""
    initial, goal = frozenset(initial), frozenset(goal)
    if goal <= initial:                       # goal already true
        return [], [initial]
    frontier = deque([initial])
    parent = {initial: None}                  # state -> (previous_state, action)
    while frontier:
        s = frontier.popleft()
        for a in actions:
            if applicable(s, a):              # logic decides what is possible
                nxt = apply_action(s, a)
                if nxt not in parent:         # avoid revisiting states
                    parent[nxt] = (s, a)
                    if goal <= nxt:           # goal test: G is a subset of the state
                        plan, states, x = [], [nxt], nxt
                        while parent[x] is not None:
                            x, act = parent[x]
                            plan.append(act); states.append(x)
                        return plan[::-1], states[::-1]
                    frontier.append(nxt)
    return None, None                         # frontier exhausted: no plan

def show_plan(plan, states):
    if plan is None:
        print("No plan found"); return
    print("S0:", sorted(states[0]))
    for i, (a, s) in enumerate(zip(plan, states[1:]), 1):
        print(f"  {i}. {a.name:<20} -> S{i}: {sorted(s)}")

In [ ]:
LOCS = ["A", "B", "C"]
CONNECTED = [("A","B"), ("B","A"), ("B","C"), ("C","B")]

def warehouse_actions(with_pickup=True):
    acts = [make_action(f"Move({x},{y})", [f"At(Robot,{x})"], (),
                        [f"At(Robot,{y})"], [f"At(Robot,{x})"]) for x, y in CONNECTED]
    if with_pickup:
        acts += [make_action(f"PickUp(Package,{l})",
                             [f"At(Robot,{l})", f"At(Package,{l})"], (),
                             ["Holding(Package)"], [f"At(Package,{l})"]) for l in LOCS]
    acts += [make_action(f"Drop(Package,{l})",
                         [f"At(Robot,{l})", "Holding(Package)"], (),
                         [f"At(Package,{l})"], ["Holding(Package)"]) for l in LOCS]
    return acts

INIT = {"At(Robot,A)", "At(Package,A)"}
GOAL = {"At(Package,C)"}

# Applicability check from Task 0
acts = warehouse_actions()
by_name = {a.name: a for a in acts}
print("PickUp(Package,A) applicable in I?", applicable(frozenset(INIT), by_name["PickUp(Package,A)"]))
print("Drop(Package,C)   applicable in I?", applicable(frozenset(INIT), by_name["Drop(Package,C)"]))

## Task 3 – Tests

In [ ]:
# Independent validator: replays a plan step by step and checks every precondition.
# It does NOT use the planner's search, so it is a separate check.
def validate_plan(init, goal, plan):
    state = frozenset(init)
    for i, a in enumerate(plan, 1):
        missing = sorted(a.pos_pre - state)
        if missing or (a.neg_pre & state):
            return False, f"Step {i} {a.name} not applicable; missing {missing}"
        state = apply_action(state, a)
    return (frozenset(goal) <= state), f"final state {sorted(state)}"

# ---- Test A: solvable ----
plan, states = bfs_plan(INIT, GOAL, warehouse_actions())
print("TEST A - solvable"); show_plan(plan, states)
print("Valid?", validate_plan(INIT, GOAL, plan))
assert [a.name for a in plan] == ["PickUp(Package,A)", "Move(A,B)", "Move(B,C)", "Drop(Package,C)"]

In [ ]:
# ---- Test B: impossible (PickUp removed) ----
plan, states = bfs_plan(INIT, GOAL, warehouse_actions(with_pickup=False))
print("TEST B - no PickUp action"); show_plan(plan, states)
assert plan is None

In [ ]:
# ---- Test C: irrelevant actions (robot moves, package does not) ----
# Goal = robot at C. Package is left behind at A.
plan, states = bfs_plan(INIT, {"At(Robot,C)"}, warehouse_actions())
print("TEST C1 - goal At(Robot,C)"); show_plan(plan, states)
print("Package at C in final state?", "At(Package,C)" in states[-1])
assert "At(Package,C)" not in states[-1]

# Goal = package at C, but only Move actions exist: robot can reach C, package cannot.
move_only = [a for a in warehouse_actions() if a.name.startswith("Move")]
plan, states = bfs_plan(INIT, GOAL, move_only)
print("\nTEST C2 - goal At(Package,C) with Move only"); show_plan(plan, states)
assert plan is None

In [ ]:
# ---- Extra tests ----
# D: goal already true -> empty plan
plan, _ = bfs_plan(INIT, {"At(Package,A)"}, warehouse_actions()); assert plan == []
print("Test D passed: goal already satisfied -> empty plan")

# E: the lab-sheet hint plan is INVALID (package is at A, not B)
by = {a.name: a for a in warehouse_actions()}
hint = [by["Move(A,B)"], by["PickUp(Package,B)"], by["Move(B,C)"], by["Drop(Package,C)"]]
ok, why = validate_plan(INIT, GOAL, hint)
print("Test E - lab hint plan valid?", ok, "|", why)
assert not ok

# F: plan found by BFS is the shortest (4 steps)
plan, _ = bfs_plan(INIT, GOAL, warehouse_actions()); assert len(plan) == 4
print("Test F passed: plan length 4 (BFS gives a shortest plan)")

## Task 4 – Logic and search
```
Current state
   ↓
Check action preconditions      (S |= Preconditions(a))   <- LOGIC
   ↓
Keep only applicable actions     <- the missing "?" step
   ↓
Generate successor state         S' = (S - delete effects) + add effects
   ↓
Search over alternatives         BFS queue of states      <- SEARCH
   ↓
Goal?  (G is a subset of S')  -> yes: return plan, no: continue
```
**Logic determines what is possible; search determines what to try.**
`applicable()` and `apply_action()` are the logic; the BFS queue in `bfs_plan()` is the search.
Without logic, search would try impossible actions. Without search, logic could say what is allowed but never choose a sequence.

## Task 5 – Can the LLM verify its own plan?
Use the validator above and compare it to the LLM's explanation (copy its explanation to a markdown cell if you want to submit it).

In [ ]:
# Independent step-by-step trace of preconditions for the plan
plan, states = bfs_plan(INIT, GOAL, warehouse_actions())
for i, a in enumerate(plan):
    print(f"{a.name}")
    print("   preconditions:", sorted(a.pos_pre))
    print("   hold in state:", a.pos_pre <= states[i], "|", sorted(states[i]))

**Which to trust?** The independently executed state transitions. They are computed mechanically from the action definitions.
An LLM explanation is generated text: it may sound right while skipping or inventing a precondition (as the lab-sheet hint plan does).
*A generated explanation is not independent verification.*

## Optional – Prolog (Tasks 6–8)
The cell below writes `planner.pl`. If SWI-Prolog is installed, the next cell runs the queries with it;
otherwise a small Python emulation gives the same answers.

In [ ]:
planner_pl = """connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :- connected(X,Y).
valid_move(X,Y) :- connected(X,Y).

% Check a whole move sequence, e.g. valid_plan([a,b,c]).
valid_plan([_]).
valid_plan([X,Y|T]) :- valid_move(X,Y), valid_plan([Y|T]).

wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
"""
open("planner.pl", "w").write(planner_pl)
print(planner_pl)

In [ ]:
import shutil, subprocess
queries = ["can_move(a,b)", "can_move(a,c)", "valid_move(a,b)", "valid_move(b,c)",
           "valid_move(a,c)", "valid_plan([a,b,c])", "valid_plan([a,c])", "reduce_speed"]

if shutil.which("swipl"):
    for q in queries:
        out = subprocess.run(["swipl", "-q", "-s", "planner.pl", "-g",
                              f"({q} -> writeln(true) ; writeln(false))", "-t", "halt"],
                             capture_output=True, text=True).stdout.strip()
        print(f"?- {q}.  ->  {out}")
else:
    print("swipl not found - using Python emulation of the same facts/rules\n")
    connected = {("a","b"), ("b","a"), ("b","c"), ("c","b")}
    can_move = valid_move = lambda x, y: (x, y) in connected
    def valid_plan(p): return all(valid_move(p[i], p[i+1]) for i in range(len(p)-1))
    results = {"can_move(a,b)": can_move("a","b"), "can_move(a,c)": can_move("a","c"),
               "valid_move(a,b)": valid_move("a","b"), "valid_move(b,c)": valid_move("b","c"),
               "valid_move(a,c)": valid_move("a","c"), "valid_plan([a,b,c])": valid_plan(["a","b","c"]),
               "valid_plan([a,c])": valid_plan(["a","c"]), "reduce_speed": True}
    for q, r in results.items():
        print(f"?- {q}.  ->  {str(r).lower()}")

**Task 6 answers**
- (a) `can_move(a,b)` is true because the fact `connected(a,b)` exists and the rule `can_move(X,Y) :- connected(X,Y)` matches.
- (b) `can_move(a,c)` fails: there is no fact `connected(a,c)` and no rule that derives it. (Prolog uses "failure to prove" = false, the closed-world assumption.)
- (c) The rule is the implication `Connected(X,Y) → CanMove(X,Y)` written right-to-left (`head :- body`). `:-` means "if".

**Task 7 challenge:** `Move(a,c)` is **not** supported: `valid_move(a,c)` fails, so the proposed action is rejected. The robot must go via b.

**Task 8:** `reduce_speed` succeeds because
`wet_road` (fact) ⇒ `slippery :- wet_road` (rule) ⇒ `reduce_speed :- slippery` (rule) ⇒ `reduce_speed` (conclusion).